**setup** (drive, kaggle, wandb)

In [ ]:
import os
from google.colab import userdata, drive

# Google Drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/fer_challenge/'
os.makedirs(SAVE_DIR, exist_ok=True)

# Kaggle
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_KEY')
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')

# WANDB
!pip install wandb -q

import wandb
wandb.login(key=userdata.get('WANDB_API_KEY'))

print("Setup completed successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akeke23 (akeke23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Setup completed successfully.


**data**

In [ ]:
!pip install -q --upgrade kaggle

!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q -o challenges-in-representation-learning-facial-expression-recognition-challenge.zip
!ls -la

print("Data ready!")

challenges-in-representation-learning-facial-expression-recognition-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)
total 978768
drwxr-xr-x 1 root root      4096 Jun 15 16:33 .
drwxr-xr-x 1 root root      4096 Jun 15 14:52 ..
-rw-r--r-- 1 root root 299063632 Dec 11  2019 challenges-in-representation-learning-facial-expression-recognition-challenge.zip
drwxr-xr-x 4 root root      4096 Jun  4 13:39 .config
drwx------ 5 root root      4096 Jun 15 15:07 drive
-rw-r--r-- 1 root root      7178 Dec 11  2019 example_submission.csv
-rw-r--r-- 1 root root  96433867 Dec 11  2019 fer2013.tar.gz
-rw-r--r-- 1 root root 301072768 Dec 11  2019 icml_face_data.csv
drwxr-xr-x 1 root root      4096 Jun  4 13:39 sample_data
-rw-r--r-- 1 root root   4801725 Jun 15 15:10 simple_cnn_best.pth
-rw-r--r-- 1 root root  60125203 Dec 11  2019 test.csv
-rw-r--r-- 1 root root 240699943 Dec 11  2019 train.csv
drwxr-xr-x 3 root root      4096 Jun 15 15:09 wandb
Data read

**gpu**

In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Active Device:", device)

Active Device: cuda


**data preprocessing**

In [ ]:
EMOTIONS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

def parse_pixels(pixel_series):
    raw_arrays = np.array([np.array(p.split(), dtype=np.uint8) for p in pixel_series])
    return raw_arrays.reshape(-1, 48, 48)

# read data and strip spaces
df = pd.read_csv('/content/icml_face_data.csv')
df.columns = df.columns.str.strip()
df['Usage'] = df['Usage'].str.strip()

# create splits
train_mask = df['Usage'] == 'Training'
val_mask   = df['Usage'] == 'PublicTest'
test_mask  = df['Usage'] == 'PrivateTest'

X_train, y_train = parse_pixels(df.loc[train_mask, 'pixels']), df.loc[train_mask, 'emotion'].values
X_val,   y_val   = parse_pixels(df.loc[val_mask, 'pixels']),   df.loc[val_mask, 'emotion'].values
X_test,  y_test  = parse_pixels(df.loc[test_mask, 'pixels']),  df.loc[test_mask, 'emotion'].values

print(f"Data Shapes - Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Data Shapes - Train: (28709, 48, 48) | Val: (3589, 48, 48) | Test: (3589, 48, 48)


In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, imgs, lbls, transform=None):
        self.imgs = imgs
        self.lbls = lbls
        self.transform = transform

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        # normalization: [0, 255] -> [0, 1]
        tensor_img = torch.from_numpy(self.imgs[idx]).float() / 255.0

        # add channel dimension: (48, 48) -> (1, 48, 48)
        tensor_img = tensor_img.unsqueeze(0)

        if self.transform:
            tensor_img = self.transform(tensor_img)
        return tensor_img, int(self.lbls[idx])

train_data = EmotionDataset(X_train, y_train)
val_data  = EmotionDataset(X_val, y_val)
test_data  = EmotionDataset(X_test, y_test)

# data streaming pipelines
train_loader = DataLoader(train_data, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_data,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Loaders ready! Batches per epoch - Train: {len(train_loader)} | Val: {len(val_loader)}")

Loaders ready! Batches per epoch - Train: 449 | Val: 57


**simpleCNN**

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.conv1_block = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2_block = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.subsampler  = nn.MaxPool2d(2, 2)
        self.dense_layer = nn.Linear(64 * 12 * 12, 128)
        self.classifier  = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.subsampler(F.relu(self.conv1_block(x)))
        x = self.subsampler(F.relu(self.conv2_block(x)))
        x = x.flatten(1)
        x = F.relu(self.dense_layer(x))
        return self.classifier(x)

In [ ]:
import math

base_model = SimpleCNN().to(device)
loss_fn = nn.CrossEntropyLoss()

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

print("FORWARD CHECK")
with torch.no_grad():
    initial_loss = loss_fn(base_model(images), labels)
print(f"Initial loss:     {initial_loss.item():.4f}")
print(f"Expected (ln7):   {math.log(7):.4f}")

print("\nBACKWARD CHECK")
optimizer = torch.optim.Adam(base_model.parameters(), lr=1e-3)
for step in range(300):
    optimizer.zero_grad()
    outputs = base_model(images)
    loss = loss_fn(outputs, labels)
    loss.backward()
    optimizer.step()
    if step % 50 == 0 or step == 299:
        acc = (outputs.argmax(1) == labels).float().mean().item()
        print(f"Step {step:3d} | Loss: {loss.item():.4f} | Accuracy: {acc:.3f}")

FORWARD CHECK
Initial loss:     1.9296
Expected (ln7):   1.9459

BACKWARD CHECK
Step   0 | Loss: 1.9296 | Accuracy: 0.344
Step  50 | Loss: 0.0035 | Accuracy: 1.000
Step 100 | Loss: 0.0005 | Accuracy: 1.000
Step 150 | Loss: 0.0002 | Accuracy: 1.000
Step 200 | Loss: 0.0001 | Accuracy: 1.000
Step 250 | Loss: 0.0000 | Accuracy: 1.000
Step 299 | Loss: 0.0000 | Accuracy: 1.000


**training**

helper functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

In [ ]:
def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

running experiment - baseline - simpleCNN

In [ ]:
def run_experiment(model, run_name, config, train_loader, val_loader, device):
    wandb.init(
        project="fer-challenge",
        name=run_name,
        group=config["architecture"],
        config=config,
        reinit=True
    )

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])

    best_val_acc = 0

    for epoch in range(1, config['epochs'] + 1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate_epoch(model, val_loader, criterion, device)

        wandb.log({
            "epoch":      epoch,
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "train_acc":  train_acc,
            "val_acc":    val_acc,
        })

        print(f"Epoch {epoch:02d}/{config['epochs']} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc

    wandb.log({"best_val_acc": best_val_acc})
    wandb.finish()
    return best_val_acc

In [ ]:
config = {
    "architecture": "SimpleCNN",
    "lr":           1e-3,
    "batch_size":   64,
    "optimizer":    "Adam",
    "epochs":       30,
}

model = SimpleCNN()
acc = run_experiment(model, "01_SimpleCNN_Baseline", config, train_loader, val_loader, device)
print(f"\nDone! Best val_acc: {acc*100:.2f}%")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 01/30 | Train Loss: 1.6451 Acc: 35.19% | Val Loss: 1.5397 Acc: 40.21%
Epoch 02/30 | Train Loss: 1.4493 Acc: 44.33% | Val Loss: 1.4067 Acc: 45.28%
Epoch 03/30 | Train Loss: 1.3408 Acc: 48.71% | Val Loss: 1.3445 Acc: 47.48%
Epoch 04/30 | Train Loss: 1.2459 Acc: 52.74% | Val Loss: 1.2972 Acc: 49.09%
Epoch 05/30 | Train Loss: 1.1738 Acc: 55.84% | Val Loss: 1.2683 Acc: 51.77%
Epoch 06/30 | Train Loss: 1.0856 Acc: 59.36% | Val Loss: 1.2953 Acc: 51.38%
Epoch 07/30 | Train Loss: 1.0059 Acc: 62.68% | Val Loss: 1.2680 Acc: 52.97%
Epoch 08/30 | Train Loss: 0.9268 Acc: 65.50% | Val Loss: 1.2937 Acc: 53.39%
Epoch 09/30 | Train Loss: 0.8452 Acc: 68.90% | Val Loss: 1.3659 Acc: 51.52%
Epoch 10/30 | Train Loss: 0.7610 Acc: 72.14% | Val Loss: 1.4097 Acc: 52.69%
Epoch 11/30 | Train Loss: 0.6842 Acc: 75.43% | Val Loss: 1.4904 Acc: 52.61%
Epoch 12/30 | Train Loss: 0.6043 Acc: 78.45% | Val Loss: 1.5357 Acc: 52.55%
Epoch 13/30 | Train Loss: 0.5264 Acc: 81.44% | Val Loss: 1.6850 Acc: 52.19%
Epoch 14/30 

best_val_acc,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇▇████████████
train_loss,█▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▄▅▆▇▇██▇███▇█▇▇▇▇▇▇▇▆▇▆▇▇▇▇▇▇
val_loss,▂▁▁▁▁▁▁▁▁▁▂▂▂▂▃▃▃▄▅▅▅▆▆▆▇▇█▇██
best_val_acc,0.53385
epoch,30
train_acc,0.97732
train_loss,0.07831
val_acc,0.51128



Done! Best val_acc: 53.39%
